In [5]:
# ============================================================
# SIMPLE DEFORESTATION DETECTION - GOOGLE COLAB
# No Authentication Required - Works Immediately!
# ============================================================

print("🌲 Installing packages... (takes 2 minutes)")

# Install required packages
!pip install -q tensorflow keras pillow opencv-python numpy matplotlib scikit-image
!pip install -q gradio  # Much easier than Flask for Colab!

print("✅ Installation complete!\n")

# ============================================================
# Import Libraries
# ============================================================

import gradio as gr
import numpy as np
import cv2
from PIL import Image
import matplotlib.pyplot as plt
from io import BytesIO
import base64

print("📚 Libraries loaded successfully!\n")

# ============================================================
# Deforestation Detection Functions
# ============================================================

def calculate_vegetation_index(image):
    """
    Calculate NDVI-like vegetation index
    NDVI = (NIR - Red) / (NIR + Red)
    For RGB images, we use: (Green - Red) / (Green + Red)
    """
    img_array = np.array(image).astype(float)

    # Extract color channels
    red = img_array[:, :, 0]
    green = img_array[:, :, 1]
    blue = img_array[:, :, 2]

    # Calculate vegetation index (green vs red ratio)
    # Avoid division by zero
    denominator = green + red + 1e-10
    vegetation_index = (green - red) / denominator

    return vegetation_index, img_array

def detect_deforestation(image):
    """
    Main function to detect deforestation in satellite imagery
    """
    # Resize image for processing
    image = image.resize((512, 512))

    # Calculate vegetation index
    veg_index, img_array = calculate_vegetation_index(image)

    # Calculate statistics
    mean_vegetation = np.mean(veg_index)
    forest_percentage = np.sum(veg_index > 0.1) / veg_index.size * 100
    deforested_percentage = 100 - forest_percentage

    # Create visualization
    fig, axes = plt.subplots(2, 2, figsize=(12, 12))

    # Original image
    axes[0, 0].imshow(image)
    axes[0, 0].set_title('Original Satellite Image', fontsize=14, fontweight='bold')
    axes[0, 0].axis('off')

    # Vegetation index heatmap
    im1 = axes[0, 1].imshow(veg_index, cmap='RdYlGn', vmin=-0.5, vmax=0.5)
    axes[0, 1].set_title('Vegetation Index (NDVI)', fontsize=14, fontweight='bold')
    axes[0, 1].axis('off')
    plt.colorbar(im1, ax=axes[0, 1], fraction=0.046)

    # Forest mask (green = forest, red = deforested)
    forest_mask = veg_index > 0.1
    color_mask = np.zeros((*veg_index.shape, 3))
    color_mask[forest_mask] = [0, 1, 0]  # Green for forest
    color_mask[~forest_mask] = [1, 0, 0]  # Red for deforested

    axes[1, 0].imshow(color_mask)
    axes[1, 0].set_title('Forest Coverage Map\n(Green=Forest, Red=Deforested)',
                         fontsize=14, fontweight='bold')
    axes[1, 0].axis('off')

    # Statistics
    axes[1, 1].axis('off')

    # Determine status
    if forest_percentage > 70:
        status = "🌳 Healthy Forest"
        status_color = 'green'
    elif forest_percentage > 40:
        status = "⚠️ Moderate Deforestation"
        status_color = 'orange'
    else:
        status = "🚨 Severe Deforestation"
        status_color = 'red'

    # Create statistics text
    stats_text = f"""
    ANALYSIS RESULTS
    {'='*40}

    Status: {status}

    Forest Coverage: {forest_percentage:.1f}%
    Deforested Area: {deforested_percentage:.1f}%

    Vegetation Index: {mean_vegetation:.3f}
    {'='*40}

    Classification:
    • > 70% forest = Healthy
    • 40-70% forest = Moderate Loss
    • < 40% forest = Severe Loss

    Analysis Method:
    Using vegetation index to detect
    healthy vegetation (shown in green)
    vs bare/degraded land (shown in red)
    """

    axes[1, 1].text(0.1, 0.5, stats_text, fontsize=12,
                    verticalalignment='center',
                    family='monospace',
                    bbox=dict(boxstyle='round', facecolor=status_color, alpha=0.3))

    plt.tight_layout()

    # Save figure to display
    buf = BytesIO()
    plt.savefig(buf, format='png', dpi=100, bbox_inches='tight')
    buf.seek(0)
    result_image = Image.open(buf)
    plt.close()

    # Create detailed report
    report = f"""
# 🌲 DEFORESTATION DETECTION REPORT

## Status: {status}

### Key Findings:
- **Forest Coverage:** {forest_percentage:.1f}%
- **Deforested Area:** {deforested_percentage:.1f}%
- **Vegetation Health Index:** {mean_vegetation:.3f}

### Analysis Details:
The system analyzed the satellite imagery using vegetation index calculation:
- **Green areas** indicate healthy forest with dense vegetation
- **Red areas** indicate deforested or degraded land
- The vegetation index ranges from -1 (no vegetation) to +1 (dense vegetation)

### Environmental Impact:
{"✅ This area shows healthy forest coverage with minimal disturbance." if forest_percentage > 70 else
 "⚠️ This area shows signs of deforestation. Monitoring recommended." if forest_percentage > 40 else
 "🚨 This area shows severe deforestation. Immediate intervention needed."}

### Methodology:
This analysis uses a simplified NDVI (Normalized Difference Vegetation Index) approach:
- Compares green vs red channels in the image
- Higher values = more vegetation
- Lower values = bare ground or deforestation

---
*Analysis powered by Computer Vision & Remote Sensing*
    """

    return result_image, report

# ============================================================
# Create Gradio Interface
# ============================================================

print("🎨 Creating web interface...\n")

# Create the Gradio interface
demo = gr.Interface(
    fn=detect_deforestation,
    inputs=gr.Image(type="pil", label="📤 Upload Satellite Image"),
    outputs=[
        gr.Image(type="pil", label="📊 Analysis Results"),
        gr.Markdown(label="📋 Detailed Report")
    ],
    title="🌲 Deforestation Detection System",
    description="""
    ### Upload satellite or aerial imagery to detect deforestation patterns

    **How it works:**
    1. Upload a satellite image (forest area, deforested area, or mixed)
    2. The system analyzes vegetation coverage using computer vision
    3. View the analysis with color-coded maps and detailed statistics

    **Best results with:**
    - Satellite imagery (Landsat, Sentinel, Google Earth)
    - Aerial photographs of forests
    - Images showing clear vegetation vs bare ground

    **Try these search terms on Google Images:**
    - "amazon deforestation satellite"
    - "forest deforestation before after"
    - "aerial forest logging"
    """,
    examples=[
        # You can add example image paths here if you have them
    ],
    theme=gr.themes.Soft(),
    allow_flagging="never"
)

# ============================================================
# Launch the App
# ============================================================

print("="*60)
print("🚀 LAUNCHING DEFORESTATION DETECTION SYSTEM")
print("="*60)
print("\n✅ App is ready!")
print("📱 The app will open in a new window")
print("🌐 You'll get a public URL you can share")
print("💡 Upload any satellite/forest image to test\n")
print("="*60 + "\n")

# Launch with public link
demo.launch(share=True, debug=False)

🌲 Installing packages... (takes 2 minutes)
✅ Installation complete!

📚 Libraries loaded successfully!

🎨 Creating web interface...



/usr/local/lib/python3.12/dist-packages/gradio/interface.py:415: UserWarning: The `allow_flagging` parameter in `Interface` is deprecated. Use `flagging_mode` instead.
  warnings.warn(


🚀 LAUNCHING DEFORESTATION DETECTION SYSTEM

✅ App is ready!
📱 The app will open in a new window
🌐 You'll get a public URL you can share
💡 Upload any satellite/forest image to test


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://1eb3e8346d8789e28d.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
